<a href="https://colab.research.google.com/github/ebubesimeon82-bit/CodeAlpha_.ipynb/blob/main/CodeAlpha_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report)

In [ ]:

# ---------------------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------------------
DATA_PATH = "Credit Risk Benchmark Dataset.csv"
df = pd.read_csv(DATA_PATH)
TARGET = "dlq_2yrs"

print("=" * 70)
print("DATA OVERVIEW")
print("=" * 70)
print(f"Shape: {df.shape}")
print(f"\nMissing values:\n{df.isna().sum()}")
print(f"\nTarget balance:\n{df[TARGET].value_counts(normalize=True)}")

DATA OVERVIEW
Shape: (16714, 11)

Missing values:
rev_util       0
age            0
late_30_59     0
debt_ratio     0
monthly_inc    0
open_credit    0
late_90        0
real_estate    0
late_60_89     0
dependents     0
dlq_2yrs       0
dtype: int64

Target balance:
dlq_2yrs
0    0.5
1    0.5
Name: proportion, dtype: float64


In [ ]:
# ---------------------------------------------------------------------------
# 2. EDA plots
# ---------------------------------------------------------------------------

figs, axes = plt.subplots(1, 2, figsize=(12, 5))
df[TARGET].value_counts().plot(kind="bar", ax=axes[0], color=["red", "green"])
axes[0].set_title("Target Class Balance (dlq_2yrs)")
axes[0].set_xticklabels(["No Delinquency (0)", "Delinquency (1)"], rotation = 0)
axes[0].set_ylabel("Count")
sns.boxplot(x=TARGET, y="rev_util", data=df[df["rev_util"] < 5], ax=axes[1])
axes[1].set_title("Revolving Utilization by Class (outliers capped for plot")
plt.tight_layout()
plt.savefig("eda_target_balance.png", dpi=150)
plt.close()
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True,fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.close()
print("\nSaved EDA plots: eda_target_balance.png, correlation_heatmap.png")



Saved EDA plots: eda_target_balance.png, correlation_heatmap.png


In [ ]:
# ---------------------------------------------------------------------------
# 3. Preprocessing
# ---------------------------------------------------------------------------
# rev_util and debt_ratio contain extreme outliers (data-entry errors, e.g.
# utilization > 1 should be rare/impossible) — cap at the 99th percentile
# instead of dropping rows, to keep the dataset size intact.
df_clean = df.copy()
for col in ["rev_util", "debt_ratio", "monthly_inc"]:
  cap = df_clean[col].quantile(0.99)
  df_clean[col] = np.where(df_clean[col] > cap, cap, df_clean[col])


In [ ]:
# ---------------------------------------------------------------------------
# 4. Feature engineering
# ---------------------------------------------------------------------------
df_clean["Total_late_payment"] = (df_clean["late_30_59"] + df_clean["late_60_89"] + df_clean["late_90"])
df_clean["income_per_dependent"] = df_clean["monthly_inc"]/(df_clean["dependents"]+1)
df_clean["credit_line_per_age"] = df_clean["open_credit"]/df_clean["age"]

feature_cols = [c for c in df_clean.columns if c != TARGET]
X = df_clean[feature_cols]
y = df_clean[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [ ]:
# ---------------------------------------------------------------------------
# 5. Train models
# ---------------------------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(random_state=42, class_weight="balanced", max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = {}
roc_dataa = {}


print("\n"+"=" * 70)
print("MODEL TRAINING & EVALUATION")
print("=" * 70)
for model_name, model in models.items():
  if model_name == "Logistic Regression":
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)
  else:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_pred_proba[:, 1])

    print(f"{model_name} Accuracy: {acc:.4f}")


MODEL TRAINING & EVALUATION
Random Forest Accuracy: 0.7718
Gradient Boosting Accuracy: 0.7783
